<a href="https://colab.research.google.com/github/SunnyChoudhary850/FlyRank/blob/main/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip -q install duckdb huggingface_hub

In [3]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

In [7]:
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, gsc_impressions
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
    ORDER BY client_hash_id, content_hash_id, report_date
    LIMIT 10
""").df()
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,report_date,gsc_impressions
0,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-01,0
1,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-02,0
2,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-03,0
3,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-04,0
4,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-05,0
5,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-06,0
6,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-07,0
7,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-08,0
8,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-09,0
9,client_0797ff3a1fc9a6a5,content_004e9c4c32e88631,2026-03-10,0


In [8]:
slice_info = con.sql(f"""
    SELECT COUNT(*) AS row_count,
           MIN(report_date) AS earliest,
           MAX(report_date) AS latest
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()
slice_info

,row_count,earliest,latest
0,9841378,2026-03-01,2026-03-31


##The Trap — Deliberate Leakage

First try: I added `impressions_next_month` as a feature, expecting the
score to jump toward perfect. It barely moved (0.65 → 0.67). Turned out my
"honest" features weren't even including `total_impressions_month`, so the
leaked column only had half the picture — it couldn't actually cheat yet.

I fixed that, but the leaked score still barely moved. Checked the label
balance first (59.6% declining, 40.4% not) — so 0.65 wasn't even much
better than just guessing the majority class. Turned out the real issue was
feature scaling: some features (raw impression counts) are way bigger
numbers than others (like days_with_activity, 0-31), and Logistic
Regression struggles to learn properly when features are on very different
scales. After scaling everything, the leak showed up clearly: honest score
stayed at 0.648, leaked score jumped to 0.875.

That jump is the leakage lesson from notebook 02, reproduced here on real
warehouse data. I removed `impressions_next_month` from my final feature
set and kept only the honest 0.648 score.

**One named limitation of my slice:** using only March 2026 as the feature
month captures one period only — content and search trends can be
seasonal, so patterns from this single month may not generalize to other
times of year.

In [9]:
availability = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()
availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


In [13]:
features = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS total_impressions_month,
           AVG(gsc_avg_position) AS avg_position_month,
           COUNT(DISTINCT report_date) AS days_with_activity,
           SUM(gsc_clicks) AS total_clicks_month,
           SUM(ga4_sessions) AS total_sessions_month
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03' AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()
features.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,total_impressions_month,avg_position_month,days_with_activity,total_clicks_month,total_sessions_month
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,1140.0,4.394234,31,2.0,0.0
1,client_73cda7b4e4f265ea,content_05597932fe4da067,57.0,2.714744,26,0.0,0.0
2,client_73cda7b4e4f265ea,content_905aa32a0230694e,149.0,6.481453,30,0.0,4.0
3,client_73cda7b4e4f265ea,content_05434271b257bb68,1421.0,6.320337,31,6.0,9.0
4,client_73cda7b4e4f265ea,content_d056587ff7faca0c,2770.0,4.459107,31,16.0,3.0
5,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,48.0,14.753175,21,0.0,0.0
6,client_73cda7b4e4f265ea,content_2662845f598544ef,150.0,6.341880,30,1.0,1.0
7,client_73cda7b4e4f265ea,content_22610b0934f8825e,67.0,12.791667,26,0.0,0.0
8,client_73cda7b4e4f265ea,content_712c365258cee05c,6048.0,4.950311,31,23.0,8.0
9,client_73cda7b4e4f265ea,content_476c37c366920c1b,223.0,50.390299,30,0.0,1.0


In [19]:
labels = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions_next_month
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-04' AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

merged = features.merge(labels, on=['client_hash_id','content_hash_id'], how='inner')
merged['declining'] = merged['impressions_next_month'] < merged['total_impressions_month']

merged = merged.dropna()

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X_honest = merged[['avg_position_month','days_with_activity','total_clicks_month','total_sessions_month']]
y = merged['declining']
X_train, X_test, y_train, y_test = train_test_split(X_honest, y, test_size=0.3, random_state=42)
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
print("Honest score:", model.score(X_test, y_test))

X_leaked = merged[['avg_position_month','days_with_activity','total_clicks_month','total_sessions_month','impressions_next_month']]
X_train2, X_test2, y_train2, y_test2 = train_test_split(X_leaked, y, test_size=0.3, random_state=42)
model2 = LogisticRegression(max_iter=1000).fit(X_train2, y_train2)
print("Leaked score (should jump way up):", model2.score(X_test2, y_test2))

Honest score: 0.6479388687199906
Leaked score (should jump way up): 0.6701311628996369
